In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
import joblib

SEED = 3042

print("Python      :", sys.version.split()[0])
print("numpy       :", np.__version__)
print("pandas      :", pd.__version__)
print("matplotlib  :", matplotlib.__version__)
print("scikit-learn:", sklearn.__version__)
print("joblib      :", joblib.__version__)
print("SEED        :", SEED)

Python      : 3.13.7
numpy       : 2.3.4
pandas      : 2.3.3
matplotlib  : 3.10.7
scikit-learn: 1.7.2
joblib      : 1.5.2
SEED        : 3042


In [2]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

print("train shape:", train.shape)
print("test shape :", test.shape)

print("\nIn train but not in test:", sorted(set(train.columns) - set(test.columns)))
print("In test but not in train:", sorted(set(test.columns) - set(train.columns)))

train.head()

train shape: (32940, 22)
test shape : (8236, 20)

In train but not in test: ['duration', 'y']
In test but not in train: []


,id,age,job,marital,education,default,housing,loan,contact,month,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,0,39,management,single,basic.9y,unknown,yes,no,telephone,may,...,3,999,0,nonexistent,1.1,93.994,-36.4,4.856,5191.0,no
1,1,32,admin.,single,university.degree,no,yes,no,cellular,aug,...,2,999,0,nonexistent,1.4,93.444,-36.1,4.963,5228.1,no
2,2,41,technician,married,high.school,no,no,yes,cellular,aug,...,1,999,0,nonexistent,1.4,93.444,-36.1,4.963,5228.1,no
3,3,39,admin.,married,professional.course,no,yes,yes,telephone,may,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,4,51,blue-collar,married,basic.9y,unknown,no,no,telephone,jun,...,1,999,0,nonexistent,1.4,94.465,-41.8,4.864,5228.1,no


## Stage 1 · Understanding the Problem

**What we predict.** We want to predict the probability that a client will agree to open a term deposit (`y`). The prediction is made before calling the client.

**The 10% rule.** The call centre can call only 10% of all clients. So, the model must rank clients from the most likely to say “yes” to the least likely. The call centre will contact only the top 10%.

**Metrics.** Accuracy is not a good metric for this task because most clients may say “no”. Instead, I use ROC-AUC and Average Precision to check how well the model finds clients who may say “yes”. For the final results, I use precision and lift in the top 10%. These metrics show how many clients say “yes” among those selected by the model.

**Train and test columns.** The `test.csv` file does not have the `y` column because the answers are hidden. It also does not have the `duration` column. Duration shows how long a call lasted, so we know it only after the call. Using this information would be data leakage because the model would use information that is not available before the call. Therefore, I remove `duration` and use only information known before calling the client.

In [3]:
print("Target counts:")
print(train["y"].value_counts())

print("\nTarget share:")
print(train["y"].value_counts(normalize=True).round(4))

always_no_acc = (train["y"] == "no").mean()
print(f"\nAccuracy of a model that always says 'no': {always_no_acc:.4f}")

Target counts:
y
no     29229
yes     3711
Name: count, dtype: int64

Target share:
y
no     0.8873
yes    0.1127
Name: proportion, dtype: float64

Accuracy of a model that always says 'no': 0.8873


## Stage 2 · Understanding the data

**Class balance.** Only 3 711 of 32 940 clients (11.27 %) said *yes*. A model that always answers *no* gets **88.73 % accuracy** while finding zero buyers, so accuracy is misleading on these data.

In [4]:
print("Real NaN values in train:", train.isna().sum().sum())

cat_cols = train.select_dtypes(include="object").columns.drop("y")
unknown_counts = (train[cat_cols] == "unknown").sum()
print("\n'unknown' values per categorical column:")
print(unknown_counts[unknown_counts > 0])

n999 = (train["pdays"] == 999).sum()
print(f"\npdays == 999: {n999} rows ({n999 / len(train):.2%})")

print("\npoutcome values:")
print(train["poutcome"].value_counts())

Real NaN values in train: 0

'unknown' values per categorical column:
job           275
marital        64
education    1401
default      6871
housing       794
loan          794
dtype: int64

pdays == 999: 31694 rows (96.22%)

poutcome values:
poutcome
nonexistent    28446
failure         3364
success         1130
Name: count, dtype: int64


**"Missing" values.** There are no real NaN values, but missing answers are coded as the category `"unknown"`: `default` 6 871 (20.9 %), `education` 1 401, `housing` 794, `loan` 794, `job` 275, `marital` 64.

**Sentinel.** `pdays = 999` means "never contacted before", not a real number of days. It appears in 31 694 rows (96.2 %). Likewise `poutcome = "nonexistent"` (28 446 rows) means "no previous campaign". Both are "not applicable" values, not measurements, and are handled in Stage 4.